In [1]:
import pandas as pd

df = pd.read_excel(
    "https://www.ers.usda.gov/media/5706/wheat-data-all-years.xlsx?v=52690",
    sheet_name="Table05",
    dtype_backend="pyarrow",
    engine="calamine",
    header=1,  # Skip the first row.
)

In [4]:
df.columns = [name.replace("/", "") for name in df.columns]
full_rows = df[~df['Beginning stocks'].isna()]
full_rows

,Marketing year 1,Time period,Beginning stocks,Production,Imports 2,Total supply 3,Food use,Seed use,Feed and residual use,Total domestic use 3,Exports 2,Total disappearance 3,Ending stocks
0,1950/51,MY Jun-May,496.0,1019.0,11.0,1526.0,580.0,--,109.0,689.0,345.0,1034.0,492.0
1,1951/52,MY Jun-May,492.0,988.0,30.0,1510.0,585.0,--,110.0,695.0,485.0,1180.0,330.0
2,1952/53,MY Jun-May,330.0,1306.0,24.0,1660.0,578.0,--,78.0,656.0,332.0,988.0,672.0
3,1953/54,MY Jun-May,672.0,1173.0,6.0,1851.0,556.0,--,87.0,643.0,214.0,857.0,994.0
4,1954/55,MY Jun-May,994.0,984.0,3.0,1981.0,552.0,--,53.0,605.0,267.0,872.0,1109.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
276,2025/26,Q1 Jun-Aug,854.734,1984.537,30.678,2869.949,240.879,2.653,239.582,483.114,252.817,735.931,2134.018
277,2025/26,Q2 Sep-Nov,2134.018,0.0,30.057,2164.075,245.577,38.521,-50.925,233.173,253.799,486.972,1677.103
278,2025/26,Q3 Dec-Feb,1677.103,0.0,32.379,1709.482,230.975,1.687,-28.831,203.831,202.678,406.509,1302.973
279,2025/26,Q4 Mar-May,1302.973,0.0,31.724,1334.697,242.569,15.039,-41.51,216.098,198.487,414.585,920.112


In [5]:
import bigframes

In [6]:
%load_ext bigframes

In [7]:
%%bqsql yearly
SELECT *
FROM {full_rows}
WHERE STARTS_WITH(`Time period`, 'MY')

,Marketing year 1,Time period,Beginning stocks,Production,Imports 2,Total supply 3,Food use,Seed use,Feed and residual use,Total domestic use 3,Exports 2,Total disappearance 3,Ending stocks
0,1955/56,MY Jun-May,1109.0,937.0,10.0,2056.0,553.0,--,51.0,604.0,322.0,926.0,1130.0
1,1957/58,MY Jun-May,1004.0,956.0,10.0,1970.0,547.0,--,43.0,590.0,418.0,1008.0,962.0
2,1954/55,MY Jun-May,994.0,984.0,3.0,1981.0,552.0,--,53.0,605.0,267.0,872.0,1109.0
3,1951/52,MY Jun-May,492.0,988.0,30.0,1510.0,585.0,--,110.0,695.0,485.0,1180.0,330.0
4,1956/57,MY Jun-May,1130.0,1005.0,8.0,2143.0,541.0,--,57.0,598.0,541.0,1139.0,1004.0
5,1950/51,MY Jun-May,496.0,1019.0,11.0,1526.0,580.0,--,109.0,689.0,345.0,1034.0,492.0
6,1962/63,MY Jun-May,1420.6,1092.0,5.3,2517.9,502.7,61.4,34.7,598.8,649.4,1248.2,1269.7
7,1959/60,MY Jun-May,1368.0,1118.0,7.0,2493.0,558.0,--,49.0,607.0,502.0,1109.0,1384.0
8,1963/64,MY Jun-May,1269.7,1146.8,4.0,2420.5,487.9,64.9,28.6,581.4,845.6,1427.0,993.5
9,1953/54,MY Jun-May,672.0,1173.0,6.0,1851.0,556.0,--,87.0,643.0,214.0,857.0,994.0


In [8]:
%%bqsql timeseries
SELECT
  * EXCEPT (`Marketing year 1`),
  TIMESTAMP(CONCAT(
    REGEXP_EXTRACT(`Marketing year 1`, r'([0-9]+)\/'),
    '-01-01')) AS `year`
FROM {yearly}


,Time period,Beginning stocks,Production,Imports 2,Total supply 3,Food use,Seed use,Feed and residual use,Total domestic use 3,Exports 2,Total disappearance 3,Ending stocks,year
0,MY Jun-May,994.0,984.0,3.0,1981.0,552.0,--,53.0,605.0,267.0,872.0,1109.0,1954-01-01 00:00:00+00:00
1,MY Jun-May,672.0,1173.0,6.0,1851.0,556.0,--,87.0,643.0,214.0,857.0,994.0,1953-01-01 00:00:00+00:00
2,MY Jun-May,1109.0,937.0,10.0,2056.0,553.0,--,51.0,604.0,322.0,926.0,1130.0,1955-01-01 00:00:00+00:00
3,MY Jun-May,1130.0,1005.0,8.0,2143.0,541.0,--,57.0,598.0,541.0,1139.0,1004.0,1956-01-01 00:00:00+00:00
4,MY Jun-May,496.0,1019.0,11.0,1526.0,580.0,--,109.0,689.0,345.0,1034.0,492.0,1950-01-01 00:00:00+00:00
5,MY Jun-May,330.0,1306.0,24.0,1660.0,578.0,--,78.0,656.0,332.0,988.0,672.0,1952-01-01 00:00:00+00:00
6,MY Jun-May,492.0,988.0,30.0,1510.0,585.0,--,110.0,695.0,485.0,1180.0,330.0,1951-01-01 00:00:00+00:00
7,MY Jun-May,1004.0,956.0,10.0,1970.0,547.0,--,43.0,590.0,418.0,1008.0,962.0,1957-01-01 00:00:00+00:00
8,MY Jun-May,962.0,1457.0,8.0,2427.0,561.0,--,48.0,609.0,450.0,1059.0,1368.0,1958-01-01 00:00:00+00:00
9,MY Jun-May,1368.0,1118.0,7.0,2493.0,558.0,--,49.0,607.0,502.0,1109.0,1384.0,1959-01-01 00:00:00+00:00


In [9]:
pddf = timeseries.set_index('year').sort_index().to_pandas()

In [14]:
pddf.to_csv("wheat.csv")